In [ ]:
import os
for variable in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"
from pathlib import Path
RUNTIME_ROOT = Path(os.environ.get("SAFETY_WATCH_RUNTIME_ROOT", "/runtime"))
PROJECT_ROOT = Path(os.environ.get("SAFETY_WATCH_PROJECT_ROOT", "/workspace"))
DATA_ROOT, MODEL_ROOT = RUNTIME_ROOT / "data", RUNTIME_ROOT / "models"
RESULT_ROOT, LOG_ROOT = RUNTIME_ROOT / "results", RUNTIME_ROOT / "logs"
SEED = 42
CALIBRATION_SEED = 20261017
VALIDATION_FRACTION = 0.20
TARGET_FPR = 0.05
CV_FOLDS = 3
REFERENCE_PARTITIONS_SHA256 = "cd5e3febd5e3025cf905214a1369344f5f22b9613fe35a2c577842eacf037c6d"
BASELINE_MODEL_CONFIG = {
    "analyzer": "char", "ngram_range": (3, 5), "lowercase": True,
    "min_df": 2, "max_features": 100_000, "sublinear_tf": True, "norm": "l2",
    "solver": "liblinear", "C": 100.0, "class_weight": "balanced", "max_iter": 1000,
    "human_weight": 1.0,
}
MODEL_CONFIG = dict(BASELINE_MODEL_CONFIG)
FEATURE_PROFILES = [
    {"min_df": 2, "max_features": 100_000, "sublinear_tf": True},
    {"min_df": 1, "max_features": 100_000, "sublinear_tf": True},
    {"min_df": 1, "max_features": 200_000, "sublinear_tf": True},
    {"min_df": 2, "max_features": 100_000, "sublinear_tf": False},
]
CANDIDATES = [
    {**BASELINE_MODEL_CONFIG, **profile, "C": c, "class_weight": weight, "human_weight": human_weight}
    for profile in FEATURE_PROFILES
    for c in (30.0, 100.0, 300.0)
    for weight in (None, "balanced")
    for human_weight in (1.0, 2.0)
]
PROTOCOL = {
    "version": "guardrail-optimization-v1", "candidate_count": 48, "cv_folds": CV_FOLDS,
    "seed": SEED, "calibration_seed": CALIBRATION_SEED, "target_calibration_fpr": TARGET_FPR,
    "reference": BASELINE_MODEL_CONFIG, "candidates": CANDIDATES,
    "ranking": "pooled human recall at <=5% inner FPR; mean AP; fewer features; smaller C; lower human weight; no class weighting; candidate order",
    "selection_scope": "fitting data only", "official_test_labels": "never read",
    "evaluation_exposure": "previously used development records; exploratory",
    "promotion_rule": "candidate recall > retrained reference and evaluation FPR <=5%; all checks pass",
}


In [ ]:
import sys, platform, json, hashlib, datetime, uuid, warnings, time, resource, shutil
import numpy as np
import pandas as pd
import scipy, sklearn, joblib, matplotlib
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, average_precision_score

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

require(sys.version_info[:2] == (3, 12), "Use the container Python 3.12 interpreter.")
require(platform.machine() == "s390x", "Training must run natively on LinuxONE s390x.")
require(sklearn.__version__ == "1.9.0", "Use the pinned scikit-learn 1.9.0 image.")
require((PROJECT_ROOT / "AGENTS.md").is_file(), "Mount repository as PROJECT_ROOT.")
require(DATA_ROOT.is_dir(), "Mount existing VM data read-only.")
environment = {
    "python": platform.python_version(), "architecture": platform.machine(),
    "interpreter": sys.executable, "sklearn": sklearn.__version__, "numpy": np.__version__,
    "scipy": scipy.__version__, "pandas": pd.__version__, "joblib": joblib.__version__,
    "matplotlib": matplotlib.__version__,
}
def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    temporary.replace(path)

run_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
run_id = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
run_dir = RESULT_ROOT / "guardrail-optimization" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
evidence_path = LOG_ROOT / "guardrail-optimization" / (run_id + ".jsonl")
evidence_path.parent.mkdir(parents=True, exist_ok=True)

def evidence(event, payload):
    previous = "0" * 64
    if evidence_path.exists():
        previous = json.loads(evidence_path.read_text().splitlines()[-1])["hash"]
    record = {"previous_hash": previous, "event": event, "payload": payload}
    record["hash"] = hashlib.sha256(canonical(record).encode()).hexdigest()
    with evidence_path.open("a") as handle:
        handle.write(canonical(record) + "\n")
    return record["hash"]

def verify_evidence(path):
    previous, count = "0" * 64, 0
    for line in Path(path).read_text().splitlines():
        record = json.loads(line)
        stored = record.pop("hash")
        require(record["previous_hash"] == previous, "Evidence link verification failed.")
        require(hashlib.sha256(canonical(record).encode()).hexdigest() == stored, "Evidence digest verification failed.")
        previous, count = stored, count + 1
    return {"records": count, "head": previous}

source_digest = sha256_file(PROJECT_ROOT / "notebooks" / "05_guardrail_optimization.ipynb")
configuration = {
    "seed": SEED, "validation_fraction": VALIDATION_FRACTION, "target_fpr": TARGET_FPR,
    "model": MODEL_CONFIG, "target": "toxicity OR jailbreaking",
    "deduplication": "exact text; exclude train/test overlaps; prefer human annotation then stable ID",
}
evidence("environment", {"environment": environment, "configuration": configuration, "notebook_sha256": source_digest})
print(json.dumps(environment, indent=2))
print("Development run:", run_id)

def choose_threshold(benign, maximum_fpr):
    benign = np.asarray(benign, dtype=float)
    require(benign.size > 0 and np.isfinite(benign).all(), "Threshold needs finite benign scores.")
    candidates = np.unique(np.concatenate(([0.0], benign, np.nextafter(benign, np.inf), [np.nextafter(1.0, np.inf)])))
    allowed = int(np.floor(maximum_fpr * benign.size + 1e-12))
    for candidate in candidates:
        if int(np.count_nonzero(benign >= candidate)) <= allowed:
            return float(candidate)
    raise RuntimeError("No threshold satisfies the FPR target.")


import subprocess
from sklearn.base import clone
git_head = (PROJECT_ROOT / ".git" / "HEAD").read_text().strip()
source_commit = ((PROJECT_ROOT / ".git" / git_head[5:]).read_text().strip()
                 if git_head.startswith("ref: ") else git_head)
protocol_digest = hashlib.sha256(canonical(PROTOCOL).encode()).hexdigest()
write_json(run_dir / "protocol.json", PROTOCOL)
evidence("protocol_locked", {"protocol_sha256": protocol_digest, "git_commit": source_commit,
                             "notebook_sha256": source_digest})
require(len(CANDIDATES) == 48, "Expected exactly 48 preregistered candidates.")


In [ ]:
manifest = json.loads((DATA_ROOT / "download-manifest.json").read_text())
entries = [entry for entry in manifest if entry.get("repo") == "lmsys/toxic-chat"]
require(len(entries) == 1, "Expected one ToxicChat manifest entry.")
dataset_entry = entries[0]
raw_root = DATA_ROOT / "raw" / "toxicchat0124"
def manifest_file(filename):
    matches = [item for item in dataset_entry["files"] if item["path"] == "data/0124/" + filename]
    require(len(matches) == 1, "Expected unique ToxicChat 0124 file.")
    item = matches[0]
    path = raw_root / item["path"]
    require(path.is_file(), "ToxicChat file missing.")
    digest = sha256_file(path)
    require(digest == item["sha256"], "Dataset SHA-256 mismatch; do not train.")
    require(path.stat().st_size == item["bytes"], "Dataset byte count mismatch.")
    return path, digest

train_path, train_digest = manifest_file("toxic-chat_annotation_train.csv")
test_path, test_digest = manifest_file("toxic-chat_annotation_test.csv")
required_columns = ["conv_id", "user_input", "human_annotation", "toxicity", "jailbreaking"]
train = pd.read_csv(train_path, usecols=required_columns, dtype={"conv_id": "string", "user_input": "string"})
require(not train[required_columns].isna().any().any(), "Missing required training fields.")
require(train["user_input"].str.strip().str.len().gt(0).all(), "Empty training prompts.")
for label in ("toxicity", "jailbreaking"):
    require(train[label].isin([0, 1]).all(), "Labels must be binary 0/1.")
train["human_annotation"] = train["human_annotation"].astype(str).map({"True": True, "False": False})
require(train["human_annotation"].notna().all(), "Invalid annotation flag.")
train["unsafe"] = (train["toxicity"].eq(1) | train["jailbreaking"].eq(1)).astype(int)
train["prompt_id"] = ["toxicchat0124:train:" + str(index) for index in range(len(train))]
train["text_sha256"] = train["user_input"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest())
dataset_metadata = {
    "repo": "lmsys/toxic-chat", "revision": dataset_entry["revision"],
    "train_sha256": train_digest, "test_sha256": test_digest,
    "official_training_rows": len(train), "license": "CC-BY-NC-4.0",
}
evidence("data_verified", dataset_metadata)
print("Official training rows:", len(train))
print(train.groupby(["human_annotation", "unsafe"]).size().rename("records").to_string())


In [ ]:
test_text = pd.read_csv(test_path, usecols=["user_input"], dtype={"user_input": "string"})
require(test_text["user_input"].notna().all(), "Missing test text.")
test_hashes = set(test_text["user_input"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()))
require(train.groupby("text_sha256")["unsafe"].nunique().le(1).all(), "Conflicting duplicate training targets.")
overlap = train["text_sha256"].isin(test_hashes)
audit_columns = ["prompt_id", "conv_id", "text_sha256", "human_annotation", "unsafe"]
excluded_overlap = train.loc[overlap, audit_columns].copy()
excluded_overlap["reason"] = "exact_test_text_overlap"
eligible = train.loc[~overlap].sort_values(
    ["human_annotation", "conv_id", "prompt_id"], ascending=[False, True, True], kind="stable"
).copy()
duplicate = eligible.duplicated("text_sha256", keep="first")
excluded_duplicate = eligible.loc[duplicate, audit_columns].copy()
excluded_duplicate["reason"] = "duplicate_training_text"
excluded = pd.concat([excluded_overlap, excluded_duplicate], ignore_index=True)
eligible = eligible.loc[~duplicate].sort_values("prompt_id", kind="stable").reset_index(drop=True)
strata = eligible["human_annotation"].astype(str) + ":" + eligible["unsafe"].astype(str)
require(strata.value_counts().ge(2).all(), "Split stratum has fewer than two records.")
fit_idx, validation_idx = train_test_split(
    np.arange(len(eligible)), test_size=VALIDATION_FRACTION, random_state=SEED, stratify=strata,
)
fit, validation = eligible.iloc[fit_idx].copy(), eligible.iloc[validation_idx].copy()
require(set(fit["prompt_id"]).isdisjoint(validation["prompt_id"]), "Fit/validation IDs overlap.")
require(set(fit["text_sha256"]).isdisjoint(validation["text_sha256"]), "Fit/validation text overlaps.")
require(set(eligible["text_sha256"]).isdisjoint(test_hashes), "Eligible training overlaps test.")
require(fit["unsafe"].nunique() == 2 and validation["unsafe"].nunique() == 2, "Each partition needs both classes.")
partitions = {
    "fit": fit[audit_columns].to_dict(orient="records"),
    "validation": validation[audit_columns].to_dict(orient="records"),
    "excluded": excluded.to_dict(orient="records"),
}
write_json(run_dir / "original-partitions.json", partitions)
partition_digest = sha256_file(run_dir / "original-partitions.json")
split_summary = {
    "cross_boundary_excluded": int(overlap.sum()), "within_training_excluded": int(duplicate.sum()),
    "eligible": len(eligible), "fit": len(fit), "validation": len(validation),
    "partitions_sha256": partition_digest,
}
evidence("partitions_locked", split_summary)
print(json.dumps(split_summary, indent=2))
del test_text, test_hashes

require(partition_digest == REFERENCE_PARTITIONS_SHA256, "Original partition membership changed.")
original_fit = fit.copy()
evaluation = validation.copy()
calibration_strata = original_fit["human_annotation"].astype(str) + ":" + original_fit["unsafe"].astype(str)
new_fit_idx, calibration_idx = train_test_split(
    np.arange(len(original_fit)), test_size=0.20, random_state=CALIBRATION_SEED,
    stratify=calibration_strata,
)
fit = original_fit.iloc[new_fit_idx].copy()
calibration = original_fit.iloc[calibration_idx].copy()
groups = {"fit": fit, "calibration": calibration, "evaluation": evaluation}
for name, frame in groups.items():
    require(frame["unsafe"].nunique() == 2, "Each partition needs both classes.")
    require(frame.loc[frame["human_annotation"], "unsafe"].nunique() == 2, "Human subset needs both classes.")
for left, right in (("fit", "calibration"), ("fit", "evaluation"), ("calibration", "evaluation")):
    require(set(groups[left]["prompt_id"]).isdisjoint(groups[right]["prompt_id"]), "Partition IDs overlap.")
    require(set(groups[left]["text_sha256"]).isdisjoint(groups[right]["text_sha256"]), "Partition texts overlap.")
new_partitions = {name: frame[audit_columns].to_dict(orient="records") for name, frame in groups.items()}
new_partitions["excluded"] = excluded.to_dict(orient="records")
write_json(run_dir / "partitions.json", new_partitions)
new_partition_digest = sha256_file(run_dir / "partitions.json")
split_summary = {
    "eligible": len(eligible), "fit": len(fit), "calibration": len(calibration), "evaluation": len(evaluation),
    "original_partitions_sha256": partition_digest, "partitions_sha256": new_partition_digest,
    "cross_boundary_excluded": int(overlap.sum()), "within_training_excluded": int(duplicate.sum()),
}
evidence("calibration_separated", split_summary)
print("New protocol:", json.dumps(split_summary, indent=2))


In [ ]:
def make_vectorizer(config):
    return TfidfVectorizer(
        analyzer="char", ngram_range=(3, 5), lowercase=True, norm="l2",
        min_df=config["min_df"], max_features=config["max_features"],
        sublinear_tf=config["sublinear_tf"],
    )

def make_classifier(config):
    return LogisticRegression(
        solver="liblinear", l1_ratio=0.0, C=config["C"],
        class_weight=config["class_weight"], max_iter=1000, random_state=SEED,
    )

def make_pipeline(config):
    return Pipeline([("tfidf", make_vectorizer(config)), ("classifier", make_classifier(config))])

def annotation_weights(frame, config):
    weights = np.where(frame["human_annotation"].to_numpy(dtype=bool), config["human_weight"], 1.0)
    require(np.isfinite(weights).all() and (weights > 0).all(), "All fitting records need positive weights.")
    return weights

search_started = time.perf_counter()
cv_strata = fit["human_annotation"].astype(str) + ":" + fit["unsafe"].astype(str)
splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
fold_results = [[] for _ in CANDIDATES]
failures = []
fold_manifest = []
for fold_number, (inner_fit_idx, inner_check_idx) in enumerate(
    splitter.split(np.zeros(len(fit)), cv_strata), start=1
):
    inner_fit, inner_check = fit.iloc[inner_fit_idx], fit.iloc[inner_check_idx]
    inner_ids, check_ids = set(inner_fit["prompt_id"]), set(inner_check["prompt_id"])
    require(inner_ids.isdisjoint(check_ids), "CV IDs overlap.")
    require((inner_ids | check_ids).isdisjoint(calibration["prompt_id"]), "Calibration leaked into CV.")
    require((inner_ids | check_ids).isdisjoint(evaluation["prompt_id"]), "Evaluation leaked into CV.")
    fold_manifest.append({"fold": fold_number, "fit_ids": inner_fit["prompt_id"].tolist(),
                          "check_ids": inner_check["prompt_id"].tolist()})
    check_human = inner_check["human_annotation"].to_numpy(dtype=bool)
    check_y = inner_check["unsafe"].to_numpy(dtype=int)
    require(np.unique(check_y[check_human]).size == 2, "Human CV check needs both classes.")
    for profile in FEATURE_PROFILES:
        vectorizer = make_vectorizer(profile)
        inner_x = vectorizer.fit_transform(inner_fit["user_input"])
        check_x = vectorizer.transform(inner_check["user_input"])
        feature_count = len(vectorizer.vocabulary_)
        indices = [index for index, config in enumerate(CANDIDATES)
                   if all(config[key] == profile[key] for key in profile)]
        for index in indices:
            config = CANDIDATES[index]
            classifier = make_classifier(config)
            started = time.perf_counter()
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("error", ConvergenceWarning)
                    classifier.fit(inner_x, inner_fit["unsafe"],
                                   sample_weight=annotation_weights(inner_fit, config))
                unsafe_index = int(np.flatnonzero(classifier.classes_ == 1)[0])
                scores = classifier.predict_proba(check_x)[:, unsafe_index]
                require(np.isfinite(scores).all(), "Nonfinite CV scores.")
                benign = scores[check_human & (check_y == 0)]
                cutoff = choose_threshold(benign, TARGET_FPR)
                unsafe = check_human & (check_y == 1)
                fold_results[index].append({
                    "fold": fold_number, "tp": int(np.count_nonzero(scores[unsafe] >= cutoff)),
                    "unsafe": int(unsafe.sum()), "fp": int(np.count_nonzero(benign >= cutoff)),
                    "benign": len(benign), "threshold": cutoff, "feature_count": feature_count,
                    "average_precision": float(average_precision_score(check_y[check_human], scores[check_human])),
                    "fit_seconds": time.perf_counter() - started,
                })
            except (ConvergenceWarning, ValueError, RuntimeError) as error:
                # Never log an exception message which could accidentally include record text.
                failures.append({"candidate": index, "fold": fold_number, "error_type": type(error).__name__})
        del inner_x, check_x, vectorizer, classifier
    evidence("cv_fold_completed", {"fold": fold_number, "candidate_count": len(CANDIDATES),
                                   "failures_so_far": len(failures)})
    print("Completed fitting-only fold", fold_number, "of", CV_FOLDS)

leaderboard = []
for index, (config, results) in enumerate(zip(CANDIDATES, fold_results)):
    if len(results) != CV_FOLDS:
        continue
    leaderboard.append({
        "candidate": index, "config": config,
        "recall": sum(row["tp"] for row in results) / sum(row["unsafe"] for row in results),
        "fpr": sum(row["fp"] for row in results) / sum(row["benign"] for row in results),
        "average_precision": float(np.mean([row["average_precision"] for row in results])),
        "mean_feature_count": float(np.mean([row["feature_count"] for row in results])),
        "folds": results,
    })
require(bool(leaderboard), "All candidates failed; retain the reference.")
chosen = max(leaderboard, key=lambda row: (
    row["recall"], row["average_precision"], -row["mean_feature_count"],
    -row["config"]["C"], -row["config"]["human_weight"],
    row["config"]["class_weight"] is None, -row["candidate"],
))
selected_config = dict(chosen["config"])
selection = {
    "selected_candidate": chosen["candidate"], "selected_config": selected_config,
    "candidate_count": len(CANDIDATES), "successful_candidates": len(leaderboard),
    "classifier_fit_attempts": len(CANDIDATES) * CV_FOLDS,
    "scope": "fitting records only", "search_seconds": time.perf_counter() - search_started,
}
write_json(run_dir / "model-selection.json", {
    "selection": selection, "leaderboard": leaderboard, "failures": failures,
    "fold_membership": fold_manifest,
})
evidence("model_selected", {**selection, "artifact_sha256": sha256_file(run_dir / "model-selection.json")})
print("Selected settings:", json.dumps(selected_config, indent=2))
print("Search seconds:", selection["search_seconds"])


In [ ]:
models, thresholds, model_digests, model_paths, training = {}, {}, {}, {}, {}
cal_human = calibration["human_annotation"].to_numpy(dtype=bool)
cal_y = calibration["unsafe"].to_numpy(dtype=int)
calibration_scores = {}
for name, config in (("reference", BASELINE_MODEL_CONFIG), ("candidate", selected_config)):
    model = make_pipeline(config)
    started = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(fit["user_input"], fit["unsafe"],
                  classifier__sample_weight=annotation_weights(fit, config))
    fit_seconds = time.perf_counter() - started
    column = int(np.flatnonzero(model.classes_ == 1)[0])
    scores = model.predict_proba(calibration["user_input"])[:, column]
    require(np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), "Invalid calibration scores.")
    cutoff = choose_threshold(scores[cal_human & (cal_y == 0)], TARGET_FPR)
    observed_fpr = float(np.mean(scores[cal_human & (cal_y == 0)] >= cutoff))
    require(observed_fpr <= TARGET_FPR + 1e-12, "Calibration FPR exceeds target.")
    model_path = run_dir / (name + ".development.joblib")
    joblib.dump(model, model_path)
    models[name], thresholds[name] = model, cutoff
    model_paths[name], model_digests[name] = model_path, sha256_file(model_path)
    calibration_scores[name] = scores
    training[name] = {
        "fit_seconds": fit_seconds, "features": len(model.named_steps["tfidf"].vocabulary_),
        "model_bytes": model_path.stat().st_size, "calibration_fpr": observed_fpr,
    }
audit_binding = {
    "protocol_sha256": protocol_digest, "partitions_sha256": new_partition_digest,
    "train_sha256": train_digest, "reference_model_sha256": model_digests["reference"],
    "candidate_model_sha256": model_digests["candidate"], "thresholds": thresholds,
    "selected_config": selected_config,
}
write_json(run_dir / "selection-before-evaluation.json", audit_binding)
evidence("models_and_thresholds_locked", {"binding": audit_binding, "training": training})
print("Calibration only:", json.dumps({"thresholds": thresholds, "training": training}, indent=2))


In [ ]:
import tempfile
def metrics(y, scores, cutoff):
    y, scores = np.asarray(y, dtype=int), np.asarray(scores, dtype=float)
    require(len(y) == len(scores) and np.unique(y).size == 2, "Metrics need both classes and matching sizes.")
    require(np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), "Invalid metric scores.")
    predicted = (scores >= cutoff).astype(int)
    tn, fp, fn, tp = (int(value) for value in confusion_matrix(y, predicted, labels=[0, 1]).ravel())
    return {
        "records": len(y), "benign": tn + fp, "unsafe": tp + fn,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "recall": tp / (tp + fn), "precision": float(precision_score(y, predicted, zero_division=0)),
        "fpr": fp / (fp + tn), "false_negative_rate": fn / (tp + fn),
        "f1": float(f1_score(y, predicted, zero_division=0)),
        "average_precision": float(average_precision_score(y, scores)),
    }

def evaluation_state(receipt_path, claim_path, binding):
    if receipt_path.exists():
        cached = json.loads(receipt_path.read_text())
        require(canonical(cached["binding"]) == canonical(binding),
                "Evaluation already belongs to different artifacts; no new scoring allowed.")
        return cached
    require(not claim_path.exists(), "Evaluation already started; inspect evidence before recovery.")
    with claim_path.open("x") as handle:
        handle.write(canonical({"binding": binding, "run_id": run_id}) + chr(10))
    return None

def must_reject(callable_check):
    try:
        callable_check()
    except RuntimeError:
        return
    raise RuntimeError("Evaluation safeguard did not reject changed state.")

with tempfile.TemporaryDirectory(prefix="receipt-check-", dir=run_dir) as directory:
    location = Path(directory)
    receipt, claim = location / "receipt.json", location / "started.json"
    require(evaluation_state(receipt, claim, audit_binding) is None and claim.exists(), "Claim not created.")
    must_reject(lambda: evaluation_state(receipt, claim, audit_binding))
    write_json(receipt, {"binding": audit_binding, "metadata_check": True})
    require(evaluation_state(receipt, claim, audit_binding)["metadata_check"], "Cached receipt not returned.")
    changed_model = {**audit_binding, "candidate_model_sha256": "0" * 64}
    must_reject(lambda: evaluation_state(receipt, claim, changed_model))
    changed_threshold = {**audit_binding, "thresholds": {**thresholds, "candidate": 0.99}}
    must_reject(lambda: evaluation_state(receipt, claim, changed_threshold))
evidence("evaluation_safeguards_verified", {
    "started_claim": True, "cache": True, "changed_model_rejected": True,
    "changed_threshold_rejected": True, "dataset_reads_in_checks": False,
})
print("Evaluation receipt safeguards passed.")


In [ ]:
evaluation_root = RESULT_ROOT / "guardrail-optimization-v1"
evaluation_root.mkdir(parents=True, exist_ok=True)
receipt_path = evaluation_root / "evaluation-receipt.json"
claim_path = evaluation_root / "evaluation-started.json"
cached = evaluation_state(receipt_path, claim_path, audit_binding)
evaluation_ids = evaluation["prompt_id"].tolist()
evaluation_scores = {}
score_path = run_dir / "evaluation-scores.jsonl"
if cached is None:
    evidence("evaluation_started", audit_binding)
    for name, model in models.items():
        column = int(np.flatnonzero(model.classes_ == 1)[0])
        evaluation_scores[name] = model.predict_proba(evaluation["user_input"])[:, column]
        require(np.isfinite(evaluation_scores[name]).all(), "Invalid evaluation scores.")
    with score_path.open("x") as handle:
        for index, prompt_id in enumerate(evaluation_ids):
            handle.write(canonical({
                "prompt_id": prompt_id, "split": "evaluation", "timestamp": run_timestamp,
                "reference_score": float(evaluation_scores["reference"][index]),
                "candidate_score": float(evaluation_scores["candidate"][index]),
            }) + chr(10))
    human = evaluation["human_annotation"].to_numpy(dtype=bool)
    labels = evaluation["unsafe"].to_numpy(dtype=int)
    scorecard = {
        name: {"human_annotated": metrics(labels[human], scores[human], thresholds[name]),
               "full_evaluation_supplementary": metrics(labels, scores, thresholds[name])}
        for name, scores in evaluation_scores.items()
    }
    receipt = {
        "binding": audit_binding, "scorecard": scorecard, "score_file": str(score_path),
        "score_file_sha256": sha256_file(score_path),
        "evaluation_ids_sha256": hashlib.sha256(canonical(evaluation_ids).encode()).hexdigest(),
        "first_run_id": run_id,
    }
    write_json(receipt_path, receipt)
    evaluation_cached = False
else:
    require(cached["evaluation_ids_sha256"] == hashlib.sha256(canonical(evaluation_ids).encode()).hexdigest(),
            "Cached evaluation IDs changed.")
    old_scores = Path(cached["score_file"])
    require(sha256_file(old_scores) == cached["score_file_sha256"], "Cached score artifact digest mismatch.")
    records = [json.loads(line) for line in old_scores.read_text().splitlines()]
    require([row["prompt_id"] for row in records] == evaluation_ids, "Cached score order changed.")
    evaluation_scores = {
        name: np.asarray([row[name + "_score"] for row in records])
        for name in ("reference", "candidate")
    }
    shutil.copyfile(old_scores, score_path)
    scorecard = cached["scorecard"]
    evaluation_cached = True
write_json(run_dir / "evaluation-scorecard.json", scorecard)
evidence("evaluation_completed_or_cached", {
    "cached": evaluation_cached, "scorecard": scorecard,
    "score_file_sha256": sha256_file(score_path),
})
print(pd.DataFrame({name: value["human_annotated"] for name, value in scorecard.items()}).T.to_string())


In [ ]:
def wilson_interval(successes, total):
    require(total > 0, "Interval denominator must be positive.")
    z = 1.959963984540054
    proportion = successes / total
    denominator = 1 + z * z / total
    center = (proportion + z * z / (2 * total)) / denominator
    radius = z * np.sqrt(proportion * (1 - proportion) / total + z * z / (4 * total * total)) / denominator
    return [float(max(0, center - radius)), float(min(1, center + radius))]

human = evaluation["human_annotation"].to_numpy(dtype=bool)
labels = evaluation["unsafe"].to_numpy(dtype=int)
unsafe = human & (labels == 1)
difference = ((evaluation_scores["candidate"][unsafe] >= thresholds["candidate"]).astype(int)
              - (evaluation_scores["reference"][unsafe] >= thresholds["reference"]).astype(int))
rng = np.random.default_rng(CALIBRATION_SEED)
sample_indices = rng.integers(0, len(difference), size=(10_000, len(difference)))
bootstrap_differences = difference[sample_indices].mean(axis=1)
paired = {
    "recall_delta": float(difference.mean()),
    "paired_bootstrap_ci95": np.quantile(bootstrap_differences, [0.025, 0.975]).tolist(),
    "bootstrap_replicates": 10_000, "unsafe_records": int(unsafe.sum()),
    "scope": "exploratory sampling interval; not correction for historical selection",
}
latency = {}
probe = calibration["user_input"].iloc[:100]
require(len(probe) == 100, "Need 100 real calibration prompts for checks.")
for name, model in models.items():
    for prompt in probe.iloc[:10]:
        model.predict_proba([prompt])
    durations = []
    for prompt in probe:
        started = time.perf_counter()
        model.predict_proba([prompt])
        durations.append((time.perf_counter() - started) * 1000)
    latency[name] = {
        "records": len(probe), "p50_ms": float(np.quantile(durations, 0.50)),
        "p95_ms": float(np.quantile(durations, 0.95)), "p99_ms": float(np.quantile(durations, 0.99)),
        "scope": "warm serial single-prompt calls on calibration records",
    }

intervals = {}
for name, scores in scorecard.items():
    value = scores["human_annotated"]
    intervals[name] = {
        "recall_ci95": wilson_interval(value["tp"], value["unsafe"]),
        "fpr_ci95": wilson_interval(value["fp"], value["benign"]),
    }
write_json(run_dir / "paired-comparison.json", {"paired": paired, "intervals": intervals})
write_json(run_dir / "latency.json", latency)
evidence("uncertainty_and_runtime_measured", {"paired": paired, "intervals": intervals, "latency": latency})
print("Paired recall difference:", json.dumps(paired, indent=2))
print("Runtime:", json.dumps(latency, indent=2))


In [ ]:
again_fit, again_calibration = train_test_split(
    np.arange(len(original_fit)), test_size=0.20, random_state=CALIBRATION_SEED,
    stratify=calibration_strata,
)
require(np.array_equal(again_fit, new_fit_idx) and np.array_equal(again_calibration, calibration_idx),
        "Calibration partition is not reproducible.")
fit_ids = set(fit["prompt_id"])
checked_ids = []
for fold in fold_manifest:
    training_ids, check_ids = set(fold["fit_ids"]), set(fold["check_ids"])
    require(training_ids.isdisjoint(check_ids) and training_ids | check_ids == fit_ids, "Invalid CV membership.")
    require((training_ids | check_ids).isdisjoint(calibration["prompt_id"]), "Calibration entered CV.")
    require((training_ids | check_ids).isdisjoint(evaluation["prompt_id"]), "Evaluation entered CV.")
    checked_ids.extend(fold["check_ids"])
require(len(checked_ids) == len(set(checked_ids)) == len(fit), "CV check folds do not cover fitting exactly once.")

for name, config in (("reference", BASELINE_MODEL_CONFIG), ("candidate", selected_config)):
    model = models[name]
    column = int(np.flatnonzero(model.classes_ == 1)[0])
    expected = model.predict_proba(probe)[:, column]
    reloaded = joblib.load(model_paths[name])
    np.testing.assert_allclose(reloaded.predict_proba(probe)[:, column], expected, rtol=0, atol=1e-12)
    repeated = make_pipeline(config)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        repeated.fit(fit["user_input"], fit["unsafe"],
                     classifier__sample_weight=annotation_weights(fit, config))
    np.testing.assert_allclose(repeated.predict_proba(probe)[:, column], expected, rtol=0, atol=1e-12)
    benign = calibration_scores[name][cal_human & (cal_y == 0)]
    tied = np.repeat(benign, 2)
    require(np.mean(tied >= choose_threshold(tied, TARGET_FPR)) <= TARGET_FPR + 1e-12, "Tied-score FPR check failed.")
    del repeated, reloaded

verify_evidence(evidence_path)
with tempfile.TemporaryDirectory(prefix="evidence-check-", dir=run_dir) as directory:
    tamper_path = Path(directory) / "tampered.jsonl"
    records = evidence_path.read_text().splitlines()
    first = json.loads(records[0])
    first["payload"]["tamper_check"] = True
    records[0] = canonical(first)
    tamper_path.write_text(chr(10).join(records) + chr(10))
    must_reject(lambda: verify_evidence(tamper_path))

evidence("verification_passed", {
    "partition_reproducibility": True, "cv_isolation": True, "reload_records_per_model": len(probe),
    "repeated_fit_records_per_model": len(probe), "score_tolerance": 1e-12,
    "threshold_ties": True, "evidence_tampering_detected": True,
})
print("Isolation, reproducibility, reload, tie, and evidence checks passed.")


In [ ]:
reference_metrics = scorecard["reference"]["human_annotated"]
candidate_metrics = scorecard["candidate"]["human_annotated"]
eligible_for_review = (candidate_metrics["recall"] > reference_metrics["recall"]
                       and candidate_metrics["fpr"] <= TARGET_FPR)
for name, config in (("reference", BASELINE_MODEL_CONFIG), ("candidate", selected_config)):
    write_json(run_dir / (name + "-model-card.json"), {
        "status": "development", "evidence_label": "verified", "verification_scope": "exploratory VM experiment",
        "environment": environment, "dataset": dataset_metadata, "config": config,
        "threshold": thresholds[name], "model_sha256": model_digests[name],
        "partitions_sha256": new_partition_digest, "notebook_sha256": source_digest,
        "source_commit": source_commit, "protocol_sha256": protocol_digest,
        "score_semantics": "unsafe score from weighted logistic regression; probability calibration unverified",
        "training": training[name], "evaluation": scorecard[name],
        "limitations": ["Prior research exposure; not fresh external evaluation.",
                        "Exact-text deduplication does not establish near-duplicate isolation.",
                        "Historical official-test-use reports conflict; no untouched-test claim.",
                        "No canary, monitoring, repair, or production validation."],
    })
decision = "eligible_for_later_review" if eligible_for_review else "retain_reference"
evidence("experiment_concluded", {
    "decision": decision, "eligible_for_review": bool(eligible_for_review),
    "frozen_models_written": False, "official_test_labels_read": False,
})
summary = {
    "run_id": run_id, "status": "exploratory", "decision": decision,
    "selected_config": selected_config, "selection": selection, "split_summary": split_summary,
    "scorecard": scorecard, "paired": paired, "intervals": intervals, "training": training,
    "latency": latency, "peak_process_rss_mib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024,
    "evaluation_cached": evaluation_cached, "protocol_sha256": protocol_digest,
    "notebook_sha256": source_digest, "source_commit": source_commit,
    "model_digests": model_digests, "thresholds": thresholds,
    "official_test_labels_read": False, "frozen_models_written": False,
    "evidence": verify_evidence(evidence_path),
}
write_json(run_dir / "run-summary.json", summary)
print(json.dumps(summary, indent=2))
